In [ ]:
# In powershell: python -m venv venv
# Activate the virtual environment: venv/Scripts/Activate.ps1
# Run: pip install -r requirements.txt to install dependencies
# Select kernel

import tensorflow as tf # version 2.18
import sklearn
from sklearn.pipeline import make_pipeline
from sklearn.compose import make_column_transformer
from sklearn.impute import SimpleImputer
from sklearn.preprocessing import OneHotEncoder, StandardScaler
from sklearn.model_selection import train_test_split
from tensorflow import keras
from tensorflow.keras import layers, callbacks, models, preprocessing, optimizers
import os, warnings
import matplotlib.pyplot as plt
from matplotlib import gridspec
import pandas as pd
import numpy as np
from sklearn.compose import make_column_selector

In [ ]:
data = pd.read_csv('path_here')
X = data.copy()
X = X.dropna(subset=['target_column_name'])  # Replace 'target_column_name' with the actual name of your target column
y = X['target_column_name']  # Replace 'target_column_name' with the actual name of your target column
X = X.drop('target_column_name', axis=1)  # Replace 'target_column_name' with the actual name of your target column
X.head()

In [ ]:
X_train, X_valid, y_train, y_valid = train_test_split(X, y, train_size=0.8, test_size=0.2, random_state=0)

In [ ]:
X.dtypes

In [ ]:
# numerical_columns = X.select_dtypes(include=['int64', 'float64']).columns
# categorical_columns = X.select_dtypes(include=['object', 'category']).columns
preprocessor = make_column_transformer(
    (
        make_pipeline(
            SimpleImputer(strategy='constant', fill_value=0),
            StandardScaler()
        ),
        make_column_selector(dtype_include=['int64', 'float64'])
    ),
    (
        make_pipeline(
            SimpleImputer(strategy='constant', fill_value='NA'),
            OneHotEncoder(handle_unknown='ignore')
        ),
        make_column_selector(dtype_include=['object', 'category'])
    )
)
X_train = preprocessor.fit_transform(X_train)
X_valid = preprocessor.transform(X_valid)

In [ ]:
model = models.Sequential([
    layers.BatchNormalization(input_shape = [X_train.shape[1]]),
    layers.Dense(units=256, activation='relu'),
    layers.BatchNormalization(),
    layers.Dropout(0.3),
    layers.Dense(units=256, activation='relu'),
    layers.BatchNormalization(),
    layers.Dropout(0.4),
    layers.Dense(units=256, activation='relu'),
    layers.BatchNormalization(),
    layers.Dropout(0.3),
    layers.Dense(units=256, activation='relu'),
    layers.BatchNormalization(),
    layers.Dropout(0.4),
    layers.Dense(units=1, activation='sigmoid')
])

In [ ]:
model.compile(optimizer='adam', loss='binary_crossentropy', metrics=['binary_accuracy'])
early_stopping = callbacks.EarlyStopping(min_delta=0.001, patience=25, restore_best_weights=True)

In [ ]:
history = model.fit(
    X_train,
    y_train,
    validation_data=(X_valid, y_valid),
    batch_size=512,
    epochs=300,
    callbacks=[early_stopping]
)

history_df = pd.DataFrame(history.history)

history_df[['loss', 'val_loss']].plot(title='Training vs Validation Loss')
history_df[['binary_accuracy', 'val_binary_accuracy']].plot(title='Training vs Validation Accuracy')

In [ ]:
from sklearn.feature_selection import mutual_info_regression

def get_mi_scores(X, y):
    X = X.copy()
    for colname in X.select_dtypes(['category', 'object']):
        X[colname], _ = X[colname].factorize()
    discrete_features = [pd.api.types.is_integer_dtype(t) for t in X.dtypes]
    scores = mutual_info_regression(X, y, discrete_features=discrete_features, random_state=0)
    scores = pd.Series(scores, name='mi_scores', index=X.columns)
    scores = scores.sort_values(ascending=False)
    return scores

get_mi_scores(X, y)

In [ ]:
from sklearn.metrics import classification_report

y_pred = model.predict(X_valid) 
y_pred = (y_pred >= 0.5).astype(int)

print(classification_report(y_valid, y_pred))